# Tamper Detection v8 (Kaggle) — Image Forgery Detection & Localization

> **v8** — Next-generation training pipeline incorporating all fixes from Run01 analysis.
> Implements pos_weight for BCE, ReduceLROnPlateau scheduler, expanded augmentation,
> per-sample Dice loss, tampered-only primary metrics, and mask-size stratified evaluation.

## Key Changes from v6.5 → v8

| Area | v6.5 | v8 |
|---|---|---|
| BCE pos_weight | Not set | Computed from training masks |
| LR Scheduler | None | ReduceLROnPlateau (patience=3, factor=0.5) |
| Dice Loss | Batch-level | Per-sample |
| Augmentation | HFlip, VFlip, Rotate90 only | + ColorJitter, ImageCompression, GaussNoise, GaussianBlur |
| Primary Metric | Mixed-set F1 (inflated) | Tampered-only F1 |
| Threshold Sweep | 0.1–0.9 (9 pts) | 0.05–0.80 (15 pts) |
| Evaluation | No size stratification | Mask-size buckets (tiny/small/medium/large) |
| cudnn.benchmark | Contradicts set_seed | Fixed — deterministic mode respected |
| Gradient Norm | Not logged | Logged per accumulation step |
| LR Logging | Not tracked | Per-epoch encoder/decoder LR logged |
| Encoder Warmup | None | Optional 2-epoch freeze |

**Architecture:** SMP U-Net (ResNet34, ImageNet pretrained)  
**Dataset:** CASIA Splicing Detection + Localization  
**Loss:** BCE (with pos_weight) + Per-Sample Dice  
**Optimizer:** AdamW (differential LR) + ReduceLROnPlateau  
**Training:** AMP, gradient accumulation, early stopping  
**Image Size:** 384 × 384 | **Split:** 70 / 15 / 15  

## Notebook Sections

1. Project Overview & Design Rationale
2. Environment Setup
3. Dataset Loading & Validation
4. Preprocessing & Augmentation
5. Model Architecture
6. Loss, Optimizer & Scheduler
7. Training Pipeline
8. Evaluation
9. Visualization
10. Explainable AI
11. Robustness Testing
12. Shortcut Learning Checks
13. Experiment Tracking
14. Save Artifacts

## 1. Design Rationale

This notebook implements the v8 training pipeline as designed in Docs8. The changes are driven by three evidence sources:

1. **Docs7** — Original system design (U-Net/ResNet34, BCE+Dice, AdamW, early stopping)
2. **Audit6 Pro** — Critical review identifying loss design gaps, metric inflation, weak augmentation
3. **Run01 Results** — Empirical evidence: tampered-only F1=0.41, threshold=0.13, copy-move F1=0.31, 13% robustness drop

### Priority Fixes Implemented

| Priority | Fix | Expected Impact |
|---|---|---|
| P0 | BCE `pos_weight` from training masks | Threshold → 0.30–0.50, tampered F1 +0.05–0.15 |
| P0 | `ReduceLROnPlateau` scheduler | Training extends from 15 to 30+ useful epochs |
| P0 | Tampered-only metrics as primary | Honest reporting of localization quality |
| P1 | Photometric + compression augmentation | Reduces JPEG robustness gap from 13% to ~5% |
| P1 | Per-sample Dice loss | Small masks get equal gradient contribution |
| P1 | Mask-size stratified evaluation | Quantifies small-region failure mode |
| P1 | Fix cudnn.benchmark contradiction | Reproducibility restored |

## 2. Environment Setup

Install dependencies, import libraries, configure the central `CONFIG` dict.

**v8 changes:**
- `pos_weight` computation added to CONFIG pipeline
- Scheduler settings added to CONFIG
- Augmentation settings exposed in CONFIG
- `cudnn.benchmark` contradiction fixed in `setup_device()`

In [1]:
!pip install -q segmentation-models-pytorch "albumentations>=1.3.1,<2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 12.1 MB/s eta 0:00:00


In [2]:
import os
import random
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

import cv2
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore', category=UserWarning)
print('All imports successful.')

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


All imports successful.


In [3]:
# ── Central Configuration ─────────────────────────────────────────────────────
# All hyperparameters and feature flags in one place.
# v8: Added scheduler, augmentation, and pos_weight settings.

SEED = 42

CONFIG = {
    # ── Data ──
    'image_size': 384,
    'batch_size': 64,
    'num_workers': 8,
    'train_ratio': 0.70,

    # ── Model ──
    'encoder_name': 'resnet34',
    'encoder_weights': 'imagenet',
    'in_channels': 3,
    'classes': 1,

    # ── Optimizer ──
    'encoder_lr': 1e-4,
    'decoder_lr': 1e-3,
    'weight_decay': 1e-4,

    # ── Scheduler (NEW in v8) ──
    'scheduler': 'reduce_on_plateau',   # 'reduce_on_plateau' | 'cosine' | 'none'
    'scheduler_patience': 3,
    'scheduler_factor': 0.5,
    'scheduler_min_lr': 1e-6,

    # ── Training ──
    'max_epochs': 50,
    'patience': 10,
    'accumulation_steps': 4,
    'max_grad_norm': 1.0,
    'encoder_warmup_epochs': 0,        # Set >0 to freeze encoder initially

    # ── Loss (NEW in v8) ──
    'use_pos_weight': True,             # Compute BCE pos_weight from training masks
    'dice_per_sample': True,            # Per-sample Dice vs batch-level
    'dice_smooth': 1.0,

    # ── Augmentation (NEW in v8) ──
    'aug_color_jitter': True,
    'aug_compression': True,
    'aug_gauss_noise': True,
    'aug_gauss_blur': True,

    # ── Feature Flags ──
    'use_amp': True,
    'use_multi_gpu': True,
    'use_wandb': True,

    # ── Reproducibility ──
    'seed': SEED,
}

# ── Kaggle output directories ──
OUTPUT_DIR = '/kaggle/working'
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')

for d in [CHECKPOINT_DIR, RESULTS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('CONFIG:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')
print(f'\nEffective batch size: {CONFIG["batch_size"] * CONFIG["accumulation_steps"]}')
print(f'Output: {OUTPUT_DIR}')

CONFIG:
  image_size: 384
  batch_size: 64
  num_workers: 8
  train_ratio: 0.7
  encoder_name: resnet34
  encoder_weights: imagenet
  in_channels: 3
  classes: 1
  encoder_lr: 0.0001
  decoder_lr: 0.001
  weight_decay: 0.0001
  scheduler: reduce_on_plateau
  scheduler_patience: 3
  scheduler_factor: 0.5
  scheduler_min_lr: 1e-06
  max_epochs: 50
  patience: 10
  accumulation_steps: 4
  max_grad_norm: 1.0
  encoder_warmup_epochs: 0
  use_pos_weight: True
  dice_per_sample: True
  dice_smooth: 1.0
  aug_color_jitter: True
  aug_compression: True
  aug_gauss_noise: True
  aug_gauss_blur: True
  use_amp: True
  use_multi_gpu: True
  use_wandb: True
  seed: 42

Effective batch size: 256
Output: /kaggle/working


In [4]:
# ── Device & Seed ─────────────────────────────────────────────────────────────

def set_seed(seed):
    """Set seed for full reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def setup_device(config):
    """Detect hardware, enable optimizations, return training device.
    
    v8 fix: Does NOT override cudnn.benchmark — set_seed() controls it.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        n_gpus = torch.cuda.device_count()

        # TF32 optimizations (does not affect determinism)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

        print(f'GPU:              {gpu_name}')
        print(f'VRAM:             {vram_gb:.1f} GB')
        print(f'GPUs available:   {n_gpus}')
        print(f'cuDNN benchmark:  {torch.backends.cudnn.benchmark}')
        print(f'TF32:             enabled')
        print(f'AMP:              {"enabled" if config["use_amp"] else "disabled"}')
        print(f'Multi-GPU:        {"enabled" if config["use_multi_gpu"] and n_gpus > 1 else "disabled"}')
    else:
        print('WARNING: No GPU detected. Training will be extremely slow.')

    print(f'Device:           {device}')
    return device


set_seed(SEED)
device = setup_device(CONFIG)

GPU:              Tesla T4
VRAM:             15.6 GB
GPUs available:   2
cuDNN benchmark:  False
TF32:             enabled
AMP:              enabled
Multi-GPU:        enabled
Device:           cuda


In [5]:
# ── W&B Experiment Tracking ───────────────────────────────────────────────────

if CONFIG['use_wandb']:
    !pip install -q wandb
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    wandb.init(
        project='v8 Tampered Image Detection & Localization',
        config=CONFIG,
        name=f'unet-resnet34-seed{SEED}-kaggle-v8',
        tags=['v8', 'casia-v2', 'kaggle', 'pos-weight', 'scheduler'],
    )
    print('W&B initialized.')
else:
    print('W&B disabled. Local artifacts only.')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: harsh_vardhan (harsh_vardhan_iiitn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run njjq5vv0
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260312_074102-njjq5vv0
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run unet-resnet34-seed42-kaggle-v8
wandb: ⭐️ View project at https://wandb.ai/harsh_vardhan_iiitn/v8%20Tampered%20Image%20Detection%20%26%20Localization
wandb: 🚀 View run at https://wandb.ai/harsh_vardhan_iiitn/v8%20Tamp

W&B initialized.


## 3. Dataset Loading & Validation

The dataset is pre-mounted by Kaggle at `/kaggle/input/`.

**Dataset:** `sagnikkayalcse52/casia-spicing-detection-localization`  
**Structure:** `Image/Au/`, `Image/Tp/` (images), `Mask/Au/`, `Mask/Tp/` (ground-truth masks)

In [6]:
# ── Kaggle Dataset Discovery ──────────────────────────────────────────────────
KAGGLE_INPUT = '/kaggle/input'

DATASET_ROOT = None
IMAGE_DIR_NAME = None
MASK_DIR_NAME = None

for root, dirs, files in os.walk(KAGGLE_INPUT):
    dirs_lower = [d.lower() for d in dirs]
    if 'image' in dirs_lower and 'mask' in dirs_lower:
        DATASET_ROOT = root
        IMAGE_DIR_NAME = dirs[dirs_lower.index('image')]
        MASK_DIR_NAME = dirs[dirs_lower.index('mask')]
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        'Could not find dataset with IMAGE/ and MASK/ directories under /kaggle/input/. '
        'Ensure the CASIA Splicing Detection + Localization dataset is attached.'
    )

print(f'Dataset root:  {DATASET_ROOT}')
print(f'Image dir:     {IMAGE_DIR_NAME}')
print(f'Mask dir:      {MASK_DIR_NAME}')

for sub in ['Au', 'Tp']:
    for parent in [IMAGE_DIR_NAME, MASK_DIR_NAME]:
        path = os.path.join(DATASET_ROOT, parent, sub)
        if os.path.exists(path):
            count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
            print(f'  {parent}/{sub}: {count} files')
        else:
            print(f'  {parent}/{sub}: NOT FOUND')

Dataset root:  /kaggle/input/datasets/sagnikkayalcse52/casia-spicing-detection-localization/New folder
Image dir:     IMAGE
Mask dir:      MASK
  IMAGE/Au: 7491 files
  MASK/Au: 7491 files
  IMAGE/Tp: 5123 files
  MASK/Tp: 5123 files


In [7]:
# ── Dataset Discovery Functions ───────────────────────────────────────────────

def validate_dimensions(image_path, mask_path):
    """Check that image and mask have the same spatial dimensions."""
    img = Image.open(image_path)
    msk = Image.open(mask_path)
    if img.size != msk.size:
        return False, f'dim_mismatch: img={img.size} mask={msk.size}'
    return True, ''


def is_valid_image(filepath):
    """Check if an image file can be opened and decoded."""
    try:
        img = Image.open(filepath)
        img.verify()
        return True
    except Exception:
        return False


def discover_pairs(dataset_root, image_dir_name, mask_dir_name):
    """Discover image-mask pairs dynamically."""
    img_tp_dir = os.path.join(dataset_root, image_dir_name, 'Tp')
    mask_tp_dir = os.path.join(dataset_root, mask_dir_name, 'Tp')
    img_au_dir = os.path.join(dataset_root, image_dir_name, 'Au')

    pairs = []
    excluded = []

    # --- Tampered images ---
    if os.path.isdir(img_tp_dir):
        for img_name in sorted(os.listdir(img_tp_dir)):
            img_path = os.path.join(img_tp_dir, img_name)
            if not os.path.isfile(img_path):
                continue

            if not is_valid_image(img_path):
                excluded.append((img_name, 'corrupt_image'))
                continue

            mask_path = os.path.join(mask_tp_dir, img_name)
            if not os.path.exists(mask_path):
                stem = Path(img_name).stem
                mask_found = False
                for ext in ['.png', '.jpg', '.bmp', '.tif']:
                    alt_mask = os.path.join(mask_tp_dir, stem + ext)
                    if os.path.exists(alt_mask):
                        mask_path = alt_mask
                        mask_found = True
                        break
                if not mask_found:
                    excluded.append((img_name, 'mask_not_found'))
                    continue

            valid, reason = validate_dimensions(img_path, mask_path)
            if not valid:
                excluded.append((img_name, reason))
                continue

            stem = Path(img_name).stem
            if '_D_' in stem:
                forgery_type = 'splicing'
            elif '_S_' in stem:
                forgery_type = 'copy-move'
            else:
                forgery_type = 'unknown'
                warnings.warn(f'Unrecognized forgery pattern: {img_name}')

            pairs.append({
                'image_path': img_path,
                'mask_path': mask_path,
                'label': 1.0,
                'forgery_type': forgery_type,
            })

    # --- Authentic images ---
    if os.path.isdir(img_au_dir):
        for img_name in sorted(os.listdir(img_au_dir)):
            img_path = os.path.join(img_au_dir, img_name)
            if not os.path.isfile(img_path):
                continue

            if not is_valid_image(img_path):
                excluded.append((img_name, 'corrupt_file'))
                continue

            pairs.append({
                'image_path': img_path,
                'mask_path': None,
                'label': 0.0,
                'forgery_type': 'authentic',
            })

    return pairs, excluded


pairs, excluded = discover_pairs(DATASET_ROOT, IMAGE_DIR_NAME, MASK_DIR_NAME)

type_counts = Counter(p['forgery_type'] for p in pairs)
print(f'Total valid pairs: {len(pairs)}')
for ftype, count in sorted(type_counts.items()):
    print(f'  {ftype}: {count}')

print(f'\nExcluded: {len(excluded)}')
if excluded:
    for name, reason in excluded[:10]:
        print(f'  {name}: {reason}')
    if len(excluded) > 10:
        print(f'  ... and {len(excluded) - 10} more')

Total valid pairs: 12614
  authentic: 7491
  copy-move: 3295
  splicing: 1828

Excluded: 0


In [8]:
# ── Validation & Stratified Splitting ─────────────────────────────────────────

assert len(pairs) > 0, 'No valid pairs discovered!'

tampered_count = sum(1 for p in pairs if p['label'] == 1.0)
authentic_count = sum(1 for p in pairs if p['label'] == 0.0)

print('=' * 50)
print('DATASET VALIDATION SUMMARY')
print('=' * 50)
print(f'Total samples:      {len(pairs) + len(excluded)}')
print(f'Valid pairs:         {len(pairs)}')
print(f'Skipped (excluded):  {len(excluded)}')
print(f'  Tampered images:   {tampered_count}')
print(f'  Authentic images:  {authentic_count}')
print(f'  Tampered ratio:    {tampered_count / len(pairs):.2%}')
print('=' * 50)

# Sample load check
print('\nSample load check:')
for i in range(min(3, len(pairs))):
    p = pairs[i]
    img = cv2.imread(p['image_path'])
    assert img is not None, f'Failed to load image: {p["image_path"]}'
    if p['mask_path'] is not None:
        mask = cv2.imread(p['mask_path'], cv2.IMREAD_GRAYSCALE)
        assert mask is not None, f'Failed to load mask: {p["mask_path"]}'
        print(f'  [{p["forgery_type"]}] img={img.shape}, mask={mask.shape}')
    else:
        print(f'  [{p["forgery_type"]}] img={img.shape}, mask=None (zero mask)')

print('\nDataset validation passed.')

DATASET VALIDATION SUMMARY
Total samples:      12614
Valid pairs:         12614
Skipped (excluded):  0
  Tampered images:   5123
  Authentic images:  7491
  Tampered ratio:    40.61%

Sample load check:
  [splicing] img=(256, 384, 3), mask=(256, 384)
  [splicing] img=(256, 384, 3), mask=(256, 384)
  [splicing] img=(256, 384, 3), mask=(256, 384)

Dataset validation passed.


In [9]:
# ── Stratified Split: 70/15/15 ────────────────────────────────────────────────

forgery_labels = [p['forgery_type'] for p in pairs]

train_pairs, temp_pairs = train_test_split(
    pairs, test_size=0.30, random_state=SEED, stratify=forgery_labels
)

temp_labels = [p['forgery_type'] for p in temp_pairs]
val_pairs, test_pairs = train_test_split(
    temp_pairs, test_size=0.5, random_state=SEED, stratify=temp_labels
)

print(f'Train: {len(train_pairs)}')
print(f'Val:   {len(val_pairs)}')
print(f'Test:  {len(test_pairs)}')

for name, split in [('Train', train_pairs), ('Val', val_pairs), ('Test', test_pairs)]:
    counts = Counter(p['forgery_type'] for p in split)
    dist = ', '.join(f'{k}: {v}' for k, v in sorted(counts.items()))
    print(f'  {name}: {dist}')

# Data leakage verification
train_paths = set(p['image_path'] for p in train_pairs)
val_paths = set(p['image_path'] for p in val_pairs)
test_paths = set(p['image_path'] for p in test_pairs)
assert len(train_paths & val_paths) == 0, 'LEAKAGE: train-val overlap!'
assert len(train_paths & test_paths) == 0, 'LEAKAGE: train-test overlap!'
assert len(val_paths & test_paths) == 0, 'LEAKAGE: val-test overlap!'
print('\nNo data leakage detected.')

Train: 8829
Val:   1892
Test:  1893
  Train: authentic: 5243, copy-move: 2306, splicing: 1280
  Val: authentic: 1124, copy-move: 494, splicing: 274
  Test: authentic: 1124, copy-move: 495, splicing: 274

No data leakage detected.


In [10]:
# ── Save Split Manifest ───────────────────────────────────────────────────────

manifest = {
    'seed': SEED,
    'total_pairs': len(pairs),
    'excluded_count': len(excluded),
    'train_count': len(train_pairs),
    'val_count': len(val_pairs),
    'test_count': len(test_pairs),
    'train': [p['image_path'] for p in train_pairs],
    'val': [p['image_path'] for p in val_pairs],
    'test': [p['image_path'] for p in test_pairs],
}

manifest_path = os.path.join(RESULTS_DIR, 'split_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Split manifest saved to: {manifest_path}')

Split manifest saved to: /kaggle/working/results/split_manifest.json


## 4. Preprocessing & Augmentation

**v8 augmentation expansion** (Docs8 §04, §02):
- `ColorJitter` — photometric regularization
- `ImageCompression` — forces model to not rely on JPEG artifacts (addresses 13% robustness gap)
- `GaussNoise` — noise robustness training
- `GaussianBlur` — blur invariance

All new augmentations are CONFIG-controlled and can be disabled individually.

In [11]:
# ── Build Augmentation Pipeline ───────────────────────────────────────────────

def build_train_transform(config):
    """Build training augmentation pipeline from CONFIG flags."""
    transforms = [
        A.Resize(config['image_size'], config['image_size']),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
    ]

    # v8: Photometric & compression augmentations
    if config.get('aug_color_jitter', False):
        transforms.append(
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5)
        )
    if config.get('aug_compression', False):
        transforms.append(
            A.ImageCompression(quality_lower=50, quality_upper=95, p=0.3)
        )
    if config.get('aug_gauss_noise', False):
        transforms.append(
            A.GaussNoise(var_limit=(10, 50), p=0.3)
        )
    if config.get('aug_gauss_blur', False):
        transforms.append(
            A.GaussianBlur(blur_limit=(3, 5), p=0.2)
        )

    transforms.extend([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

    return A.Compose(transforms)


train_transform = build_train_transform(CONFIG)

val_transform = A.Compose([
    A.Resize(CONFIG['image_size'], CONFIG['image_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print('Train augmentations:')
for t in train_transform.transforms:
    print(f'  {t.__class__.__name__}')
print(f'\nVal/Test: Resize {CONFIG["image_size"]}, Normalize')

Train augmentations:
  Resize
  HorizontalFlip
  VerticalFlip
  RandomRotate90
  ColorJitter
  ImageCompression
  GaussNoise
  GaussianBlur
  Normalize
  ToTensorV2

Val/Test: Resize 384, Normalize


In [12]:
# ── Dataset Class ─────────────────────────────────────────────────────────────

class TamperingDataset(Dataset):
    """Dataset for tamper detection with segmentation masks."""

    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        entry = self.pairs[idx]

        image = cv2.imread(entry['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if entry['mask_path'] is not None:
            mask = cv2.imread(entry['mask_path'], cv2.IMREAD_GRAYSCALE)
            mask = (mask > 0).astype(np.uint8)
        else:
            h, w = image.shape[:2]
            mask = np.zeros((h, w), dtype=np.uint8)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask)
        mask = mask.unsqueeze(0).float()

        label = torch.tensor(entry['label'], dtype=torch.float32)
        return image, mask, label


print('TamperingDataset defined.')

TamperingDataset defined.


In [13]:
# ── DataLoaders ───────────────────────────────────────────────────────────────

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

train_dataset = TamperingDataset(train_pairs, transform=train_transform)
val_dataset = TamperingDataset(val_pairs, transform=val_transform)
test_dataset = TamperingDataset(test_pairs, transform=val_transform)

_nw = CONFIG['num_workers']
loader_kwargs = dict(
    num_workers=_nw,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=_nw > 0,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True,
    worker_init_fn=seed_worker,
    generator=g,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')
print(f'num_workers={_nw}, pin_memory={loader_kwargs["pin_memory"]}, persistent_workers={loader_kwargs["persistent_workers"]}')

Train batches: 137
Val batches:   30
Test batches:  30
num_workers=8, pin_memory=True, persistent_workers=True


In [14]:
# ── Sanity Check: Visualize One Training Batch ────────────────────────────────

images, masks, labels = next(iter(train_loader))
print(f'Image batch: {images.shape}  Mask batch: {masks.shape}  Labels: {labels}')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(min(4, images.size(0))):
    img = images[i].permute(1, 2, 0).numpy() * std + mean
    img = np.clip(img, 0, 1)
    msk = masks[i].squeeze().numpy()

    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image (label={labels[i].item():.0f})')
    axes[0, i].axis('off')

    axes[1, i].imshow(msk, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title('Mask')
    axes[1, i].axis('off')

plt.suptitle('Sanity Check: Training Batch (with v8 augmentations)')
plt.tight_layout()
plt.show()

Image batch: torch.Size([64, 3, 384, 384])  Mask batch: torch.Size([64, 1, 384, 384])  Labels: tensor([1., 1., 1., 0., 0., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0.,
        0., 1., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1.,
        1., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 1., 1., 1., 1., 1.])


## 5. Model Architecture

**SMP U-Net** with ResNet34 encoder (ImageNet pretrained). ~24M parameters.

Architecture is retained from v6.5 — Docs8 §03 determined that the architecture is not the primary bottleneck (loss design and augmentation are). SMP makes future encoder swaps trivial.

The model outputs raw logits (no activation). Sigmoid is applied in the loss and evaluation.

In [15]:
# ── Model Setup ───────────────────────────────────────────────────────────────

def setup_model(config, device):
    """Create model, optionally wrap in DataParallel, verify output shape."""
    model = smp.Unet(
        encoder_name=config['encoder_name'],
        encoder_weights=config['encoder_weights'],
        in_channels=config['in_channels'],
        classes=config['classes'],
        activation=None,
    )
    model = model.to(device)

    is_parallel = False
    if torch.cuda.device_count() > 1 and config['use_multi_gpu']:
        model = torch.nn.DataParallel(model)
        is_parallel = True
        print(f'DataParallel enabled across {torch.cuda.device_count()} GPUs.')
    else:
        print('Single-device training.')

    total_params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters:     {total_params:,}')
    print(f'Trainable parameters: {trainable:,}')

    with torch.no_grad():
        dummy = torch.randn(1, 3, config['image_size'], config['image_size']).to(device)
        out = model(dummy)
        assert out.shape == (1, 1, config['image_size'], config['image_size']), \
            f'Unexpected output shape: {out.shape}'
    print(f'Shape check passed: (1, 3, {config["image_size"]}, {config["image_size"]}) -> {out.shape}')

    return model, is_parallel


model, is_parallel = setup_model(CONFIG, device)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

DataParallel enabled across 2 GPUs.
Total parameters:     24,436,369
Trainable parameters: 24,436,369
Shape check passed: (1, 3, 384, 384) -> torch.Size([1, 1, 384, 384])


## 6. Loss, Optimizer & Scheduler

**v8 critical changes** (Docs8 §04, §08):

1. **BCE `pos_weight`** — Computed from training masks. Addresses threshold=0.13 anomaly from Run01.
2. **Per-sample Dice** — Each image contributes equally to Dice gradient, regardless of mask size.
3. **ReduceLROnPlateau** — Reduces LR when val F1 stalls. Prevents overfitting after convergence.

Loss alternatives (Focal, Tversky) are structured for easy substitution.

In [16]:
# ── Compute pos_weight from Training Masks (v8 P0 Fix) ────────────────────────

pos_weight = None

if CONFIG['use_pos_weight']:
    print('Computing pos_weight from training masks...')
    total_fg, total_bg = 0, 0
    for pair in tqdm(train_pairs, desc='Scanning masks'):
        if pair['mask_path'] is not None:
            mask = cv2.imread(pair['mask_path'], cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                fg = (mask > 0).sum()
                total_fg += fg
                total_bg += mask.size - fg
        else:
            # Authentic image: all background (use average image size estimate)
            total_bg += CONFIG['image_size'] ** 2

    pw_value = total_bg / max(total_fg, 1)
    pos_weight = torch.tensor([pw_value], dtype=torch.float32).to(device)
    print(f'pos_weight: {pw_value:.2f} (bg_pixels={total_bg:,}, fg_pixels={total_fg:,})')
    print(f'Foreground fraction: {total_fg / (total_fg + total_bg):.4%}')
else:
    print('pos_weight disabled (CONFIG["use_pos_weight"] = False)')

Computing pos_weight from training masks...


Scanning masks:   0%|          | 0/8829 [00:00<?, ?it/s]

pos_weight: 30.01 (bg_pixels=1,328,847,646, fg_pixels=44,277,628)
Foreground fraction: 3.2246%


In [17]:
# ── Loss Function (v8: per-sample Dice + pos_weight BCE) ─────────────────────

class BCEDiceLoss(nn.Module):
    """Combined BCE + Dice loss for binary segmentation.
    
    v8 changes:
    - Supports pos_weight in BCE for class imbalance correction
    - Per-sample Dice computation to prevent large-mask gradient dominance
    """

    def __init__(self, pos_weight=None, smooth=1.0, per_sample_dice=True):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.smooth = smooth
        self.per_sample_dice = per_sample_dice

    def _dice_loss_per_sample(self, logits, targets):
        """Compute Dice loss per sample, then average."""
        probs = torch.sigmoid(logits)
        batch_size = probs.shape[0]
        losses = []
        for i in range(batch_size):
            p = probs[i].reshape(-1)
            t = targets[i].reshape(-1)
            intersection = (p * t).sum()
            dice = (2.0 * intersection + self.smooth) / (
                p.sum() + t.sum() + self.smooth
            )
            losses.append(1.0 - dice)
        return torch.stack(losses).mean()

    def _dice_loss_batch(self, logits, targets):
        """Compute Dice loss across entire batch (v6.5 behavior)."""
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum()
        return 1.0 - (2.0 * intersection + self.smooth) / (
            probs.sum() + targets.sum() + self.smooth
        )

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        if self.per_sample_dice:
            dice_loss = self._dice_loss_per_sample(logits, targets)
        else:
            dice_loss = self._dice_loss_batch(logits, targets)
        return bce_loss + dice_loss


# --- Alternative loss functions (structured for easy experimentation) ---

class FocalDiceLoss(nn.Module):
    """Focal Loss + Dice for hard example mining. Drop-in replacement."""

    def __init__(self, alpha=1.0, gamma=2.0, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        # Focal loss
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        focal_loss = focal.mean()

        # Per-sample Dice
        probs = torch.sigmoid(logits)
        losses = []
        for i in range(probs.shape[0]):
            p, t = probs[i].reshape(-1), targets[i].reshape(-1)
            intersection = (p * t).sum()
            dice = (2.0 * intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)
            losses.append(1.0 - dice)
        dice_loss = torch.stack(losses).mean()

        return focal_loss + dice_loss


class TverskyDiceLoss(nn.Module):
    """Tversky Loss + Dice. Adjustable FP/FN balance via alpha/beta."""

    def __init__(self, alpha=0.3, beta=0.7, smooth=1.0):
        super().__init__()
        self.alpha = alpha  # FP weight (lower = tolerate more FP)
        self.beta = beta    # FN weight (higher = penalize missed detections)
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        losses = []
        for i in range(probs.shape[0]):
            p, t = probs[i].reshape(-1), targets[i].reshape(-1)
            tp = (p * t).sum()
            fp = (p * (1 - t)).sum()
            fn = ((1 - p) * t).sum()
            tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
            losses.append(1.0 - tversky)
        return torch.stack(losses).mean()


# --- Instantiate loss ---
criterion = BCEDiceLoss(
    pos_weight=pos_weight,
    smooth=CONFIG['dice_smooth'],
    per_sample_dice=CONFIG['dice_per_sample'],
)

print(f'Loss: BCEDiceLoss (pos_weight={"enabled" if pos_weight is not None else "disabled"}, '
      f'dice={"per-sample" if CONFIG["dice_per_sample"] else "batch"})')

Loss: BCEDiceLoss (pos_weight=enabled, dice=per-sample)


In [18]:
# ── Optimizer & Scheduler (v8: ReduceLROnPlateau) ─────────────────────────────

base_model = model.module if is_parallel else model

optimizer = torch.optim.AdamW([
    {'params': base_model.encoder.parameters(), 'lr': CONFIG['encoder_lr']},
    {'params': base_model.decoder.parameters(), 'lr': CONFIG['decoder_lr']},
    {'params': base_model.segmentation_head.parameters(), 'lr': CONFIG['decoder_lr']},
], weight_decay=CONFIG['weight_decay'])

# v8: Learning rate scheduler
scheduler = None
if CONFIG['scheduler'] == 'reduce_on_plateau':
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=CONFIG['scheduler_factor'],
        patience=CONFIG['scheduler_patience'],
        min_lr=CONFIG['scheduler_min_lr'],
    )
    print(f'Scheduler: ReduceLROnPlateau (patience={CONFIG["scheduler_patience"]}, '
          f'factor={CONFIG["scheduler_factor"]}, min_lr={CONFIG["scheduler_min_lr"]})')
elif CONFIG['scheduler'] == 'cosine':
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2
    )
    print('Scheduler: CosineAnnealingWarmRestarts (T_0=10, T_mult=2)')
else:
    print('Scheduler: None')

# Mixed precision scaler
scaler = GradScaler('cuda', enabled=CONFIG['use_amp'])

print(f'Optimizer: AdamW (encoder_lr={CONFIG["encoder_lr"]}, decoder_lr={CONFIG["decoder_lr"]})')
print(f'AMP scaler enabled: {CONFIG["use_amp"]}')

Scheduler: ReduceLROnPlateau (patience=3, factor=0.5, min_lr=1e-06)
Optimizer: AdamW (encoder_lr=0.0001, decoder_lr=0.001)
AMP scaler enabled: True


## 7. Training Pipeline

**v8 changes from v6.5:**
- Scheduler step after each validation epoch
- Gradient norm logged per accumulation step
- LR per param group logged to W&B
- Optional encoder warmup (freeze for N epochs)
- Early stopping still on val Pixel-F1 (patience=10)

In [19]:
# ── Evaluation Metrics ────────────────────────────────────────────────────────

def compute_pixel_f1(pred, gt, eps=1e-8):
    """Compute Pixel-F1 score."""
    pred, gt = pred.flatten(), gt.flatten()
    if gt.sum() == 0 and pred.sum() == 0:
        return 1.0
    if gt.sum() == 0 and pred.sum() > 0:
        return 0.0
    tp = (pred * gt).sum()
    fp = (pred * (1 - gt)).sum()
    fn = ((1 - pred) * gt).sum()
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    return (2 * precision * recall / (precision + recall + eps)).item()


def compute_iou(pred, gt, eps=1e-8):
    """Compute Intersection over Union."""
    pred, gt = pred.flatten(), gt.flatten()
    if gt.sum() == 0 and pred.sum() == 0:
        return 1.0
    intersection = (pred * gt).sum()
    union = pred.sum() + gt.sum() - intersection
    if union == 0:
        return 1.0
    return (intersection / (union + eps)).item()


def compute_precision_recall(pred, gt, eps=1e-8):
    """Compute pixel-level precision and recall."""
    pred, gt = pred.flatten(), gt.flatten()
    if gt.sum() == 0 and pred.sum() == 0:
        return 1.0, 1.0
    if gt.sum() == 0 and pred.sum() > 0:
        return 0.0, 1.0
    tp = (pred * gt).sum()
    fp = (pred * (1 - gt)).sum()
    fn = ((1 - pred) * gt).sum()
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    return precision.item(), recall.item()


print('Metric functions defined.')

Metric functions defined.


In [20]:
# ── Training & Validation Helpers (v8: gradient norm logging) ─────────────────

def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, config,
                    log_grad_norm=False):
    """Run one training epoch with gradient accumulation, optional AMP, and grad norm logging."""
    model.train()
    running_loss = 0.0
    accum_steps = config['accumulation_steps']
    use_amp = config['use_amp']
    grad_norms = []
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), leave=False)

    for batch_idx, (images, masks, labels) in pbar:
        images = images.to(device)
        masks = masks.to(device)

        with autocast('cuda', enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, masks) / accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            total_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=config['max_grad_norm']
            )
            if log_grad_norm:
                grad_norms.append(total_norm.item())
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * accum_steps
        pbar.set_postfix({'loss': f'{running_loss / (batch_idx + 1):.4f}'})

    # Partial window flush
    if (batch_idx + 1) % accum_steps != 0:
        scaler.unscale_(optimizer)
        total_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=config['max_grad_norm']
        )
        if log_grad_norm:
            grad_norms.append(total_norm.item())
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    avg_loss = running_loss / len(train_loader)
    avg_grad_norm = np.mean(grad_norms) if grad_norms else 0.0
    return avg_loss, avg_grad_norm


@torch.no_grad()
def validate_model(model, val_loader, criterion, device, config, threshold=0.5):
    """Validation with optional AMP. Returns loss, mean F1, mean IoU."""
    model.eval()
    total_loss = 0.0
    f1_scores = []
    iou_scores = []
    use_amp = config['use_amp']

    for images, masks, labels in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        with autocast('cuda', enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, masks)

        total_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs > threshold).float()

        for i in range(images.size(0)):
            f1_scores.append(compute_pixel_f1(preds[i], masks[i]))
            iou_scores.append(compute_iou(preds[i], masks[i]))

    avg_loss = total_loss / len(val_loader.dataset)
    avg_f1 = np.mean(f1_scores)
    avg_iou = np.mean(iou_scores)
    return avg_loss, avg_f1, avg_iou


print('train_one_epoch() and validate_model() defined.')

train_one_epoch() and validate_model() defined.


In [21]:
# ── Checkpoint Helpers ────────────────────────────────────────────────────────

def save_checkpoint(state, filepath):
    torch.save(state, filepath)


def load_checkpoint(filepath, model, optimizer, scaler, device, is_parallel,
                    scheduler=None):
    ckpt = torch.load(filepath, map_location=device, weights_only=False)
    state_dict = ckpt['model_state_dict']

    if is_parallel and not any(k.startswith('module.') for k in state_dict):
        state_dict = {f'module.{k}': v for k, v in state_dict.items()}
    elif not is_parallel and any(k.startswith('module.') for k in state_dict):
        state_dict = {k.replace('module.', '', 1): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    return ckpt['epoch'] + 1, ckpt['best_f1'], ckpt['best_epoch']


print('Checkpoint helpers defined.')

Checkpoint helpers defined.


In [22]:
# ── Training Loop (v8: scheduler, grad norm, LR logging, encoder warmup) ─────

history = {
    'train_loss': [],
    'val_loss': [],
    'val_f1': [],
    'val_iou': [],
    'lr_encoder': [],
    'lr_decoder': [],
    'grad_norm': [],
}

best_f1 = 0.0
best_epoch = 0
start_epoch = 0

# Optional: resume from checkpoint
resume_path = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt')
if os.path.exists(resume_path):
    start_epoch, best_f1, best_epoch = load_checkpoint(
        resume_path, model, optimizer, scaler, device, is_parallel, scheduler
    )
    print(f'Resumed from epoch {start_epoch}, best F1={best_f1:.4f} at epoch {best_epoch + 1}')

# v8: Encoder warmup — freeze encoder for first N epochs
warmup_epochs = CONFIG.get('encoder_warmup_epochs', 0)
if warmup_epochs > 0:
    print(f'Encoder warmup: freezing encoder for first {warmup_epochs} epochs')
    for param in base_model.encoder.parameters():
        param.requires_grad = False

for epoch in range(start_epoch, CONFIG['max_epochs']):
    # v8: Unfreeze encoder after warmup
    if warmup_epochs > 0 and epoch == warmup_epochs:
        print(f'  Encoder unfrozen at epoch {epoch + 1}')
        for param in base_model.encoder.parameters():
            param.requires_grad = True

    print(f'\nEpoch {epoch + 1}/{CONFIG["max_epochs"]}')

    # Train
    train_loss, avg_grad_norm = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device, CONFIG,
        log_grad_norm=True,
    )

    # Validate
    val_loss, val_f1, val_iou = validate_model(
        model, val_loader, criterion, device, CONFIG
    )

    # v8: Scheduler step
    if scheduler is not None:
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_f1)
        else:
            scheduler.step()

    # Current learning rates
    lr_enc = optimizer.param_groups[0]['lr']
    lr_dec = optimizer.param_groups[1]['lr']

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_iou'].append(val_iou)
    history['lr_encoder'].append(lr_enc)
    history['lr_decoder'].append(lr_dec)
    history['grad_norm'].append(avg_grad_norm)

    print(
        f'  train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, '
        f'val_f1={val_f1:.4f}, val_iou={val_iou:.4f}'
    )
    print(f'  LR: encoder={lr_enc:.2e}, decoder={lr_dec:.2e}, grad_norm={avg_grad_norm:.4f}')

    # W&B logging
    if CONFIG['use_wandb']:
        wandb.log({
            'epoch': epoch + 1,
            'train/loss': train_loss,
            'val/loss': val_loss,
            'val/pixel_f1': val_f1,
            'val/pixel_iou': val_iou,
            'train/lr_encoder': lr_enc,
            'train/lr_decoder': lr_dec,
            'train/grad_norm': avg_grad_norm,
        })

    # Checkpoint
    model_state = model.module.state_dict() if is_parallel else model.state_dict()
    state = {
        'epoch': epoch,
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'best_f1': best_f1,
        'best_epoch': best_epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_f1': val_f1,
    }

    save_checkpoint(state, os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt'))

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        state['best_f1'] = best_f1
        state['best_epoch'] = best_epoch
        save_checkpoint(state, os.path.join(CHECKPOINT_DIR, 'best_model.pt'))
        print(f'  -> New best model saved (F1={best_f1:.4f})')

    if (epoch + 1) % 10 == 0:
        save_checkpoint(state, os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch + 1}.pt'))

    if epoch - best_epoch >= CONFIG['patience']:
        print(f'Early stopping at epoch {epoch + 1}. Best F1={best_f1:.4f} at epoch {best_epoch + 1}')
        break

print(f'\nTraining complete. Best F1={best_f1:.4f} at epoch {best_epoch + 1}')


Epoch 1/50


  0%|          | 0/137 [00:00<?, ?it/s]

  train_loss=2.2359, val_loss=2.1599, val_f1=0.0826, val_iou=0.0555
  LR: encoder=1.00e-04, decoder=1.00e-03, grad_norm=2.0677
  -> New best model saved (F1=0.0826)

Epoch 2/50


  0%|          | 0/137 [00:00<?, ?it/s]

  train_loss=2.1007, val_loss=2.1086, val_f1=0.0969, val_iou=0.0689
  LR: encoder=1.00e-04, decoder=1.00e-03, grad_norm=2.2994
  -> New best model saved (F1=0.0969)

Epoch 3/50


...

  train_loss=1.4293, val_loss=2.3081, val_f1=0.3383, val_iou=0.3052
  LR: encoder=1.25e-05, decoder=1.25e-04, grad_norm=3.8846

Epoch 26/50


  0%|          | 0/137 [00:00<?, ?it/s]

  train_loss=1.4203, val_loss=2.2874, val_f1=0.3392, val_iou=0.3059
  LR: encoder=1.25e-05, decoder=1.25e-04, grad_norm=3.9256

Epoch 27/50


  0%|          | 0/137 [00:00<?, ?it/s]

  train_loss=1.4455, val_loss=2.2630, val_f1=0.3198, val_iou=0.2862
  LR: encoder=1.25e-05, decoder=1.25e-04, grad_norm=3.9937
Early stopping at epoch 27. Best F1=0.3585 at epoch 17

Training complete. Best F1=0.3585 at epoch 17


## 8. Evaluation

**v8 evaluation improvements** (Docs8 §05):
1. **Tampered-only metrics reported FIRST** — primary indicator of localization quality
2. **Expanded threshold sweep** — 0.05 to 0.80 in 0.05 steps (15 candidates vs v6.5's 9)
3. **Mask-size stratified analysis** — tiny (<2%), small (2-5%), medium (5-15%), large (>15%)
4. **Per-forgery-type breakdown** in main output

In [23]:
# Load best model for evaluation
best_ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)

base_model = model.module if is_parallel else model
base_model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {ckpt["best_epoch"] + 1} (F1={ckpt["best_f1"]:.4f})')

Loaded best model from epoch 17 (F1=0.3585)


In [24]:
# ── Threshold Selection (v8: expanded sweep) ──────────────────────────────────

@torch.no_grad()
def find_best_threshold(model, val_loader, device, config, thresholds=None):
    """Sweep thresholds on validation set to maximize mean Pixel-F1.
    
    v8: Default sweep range 0.05-0.80 in 0.05 steps (was 0.1-0.9 in v6.5).
    """
    model.eval()
    if thresholds is None:
        thresholds = np.arange(0.05, 0.80, 0.05)

    all_probs = []
    all_masks = []
    for images, masks, labels in tqdm(val_loader, desc='Collecting val predictions'):
        images = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            logits = model(images)
        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)
        all_masks.append(masks)

    all_probs = torch.cat(all_probs)
    all_masks = torch.cat(all_masks)

    results = []
    for t in tqdm(thresholds, desc='Threshold sweep'):
        f1_scores = []
        preds = (all_probs > t).float()
        for i in range(len(all_probs)):
            f1_scores.append(compute_pixel_f1(preds[i], all_masks[i]))
        results.append((t, np.mean(f1_scores)))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[0][0], results


best_threshold, threshold_results = find_best_threshold(model, val_loader, device, CONFIG)
print(f'Best threshold: {best_threshold:.4f}')
print(f'Best val F1 at threshold: {threshold_results[0][1]:.4f}')

# v8 calibration check
if best_threshold < 0.20:
    print(f'\n⚠ WARNING: Threshold {best_threshold:.4f} is below 0.20.')
    print('  This suggests probability calibration may still need improvement.')
    print('  Expected range with pos_weight: 0.25–0.55')
elif best_threshold > 0.55:
    print(f'\n⚠ NOTE: Threshold {best_threshold:.4f} is above 0.55.')
    print('  pos_weight may be too aggressive. Consider reducing.')
else:
    print(f'\n✓ Threshold {best_threshold:.4f} is in expected range (0.20–0.55).')

Threshold sweep:   0%|          | 0/15 [00:00<?, ?it/s]

Best threshold: 0.7500
Best val F1 at threshold: 0.5198

⚠ NOTE: Threshold 0.7500 is above 0.55.
  pos_weight may be too aggressive. Consider reducing.


In [25]:
# ── Full Test Evaluation (v8: mask-size stratification) ───────────────────────

@torch.no_grad()
def evaluate(model, test_loader, test_pairs, device, config, threshold):
    """Full test evaluation with mask-size stratification (v8)."""
    model.eval()

    all_f1, all_iou, all_precision, all_recall = [], [], [], []
    tampered_f1, tampered_iou = [], []
    image_preds, image_labels, image_scores = [], [], []
    forgery_f1 = {'splicing': [], 'copy-move': []}

    # v8: Mask-size stratification
    size_buckets = {
        'tiny (<2%)': {'range': (0, 0.02), 'f1': []},
        'small (2-5%)': {'range': (0.02, 0.05), 'f1': []},
        'medium (5-15%)': {'range': (0.05, 0.15), 'f1': []},
        'large (>15%)': {'range': (0.15, 1.0), 'f1': []},
    }

    idx = 0
    for images, masks, labels in tqdm(test_loader, desc='Evaluating'):
        images = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            logits = model(images)
        probs = torch.sigmoid(logits).cpu()
        preds = (probs > threshold).float()

        for i in range(images.size(0)):
            f1 = compute_pixel_f1(preds[i], masks[i])
            iou = compute_iou(preds[i], masks[i])
            prec, rec = compute_precision_recall(preds[i], masks[i])

            all_f1.append(f1)
            all_iou.append(iou)
            all_precision.append(prec)
            all_recall.append(rec)

            tamper_score = probs[i].view(-1).max().item()
            image_scores.append(tamper_score)
            image_labels.append(int(labels[i].item()))
            image_preds.append(int(tamper_score >= threshold))

            if labels[i].item() == 1.0:
                tampered_f1.append(f1)
                tampered_iou.append(iou)

                # Forgery type
                if idx < len(test_pairs):
                    ftype = test_pairs[idx]['forgery_type']
                    if ftype in forgery_f1:
                        forgery_f1[ftype].append(f1)

                # v8: Mask-size bucket
                mask_ratio = masks[i].sum().item() / masks[i].numel()
                for bucket_name, bucket_info in size_buckets.items():
                    lo, hi = bucket_info['range']
                    if lo <= mask_ratio < hi:
                        bucket_info['f1'].append(f1)
                        break

            idx += 1

    image_accuracy = np.mean([p == l for p, l in zip(image_preds, image_labels)])
    try:
        image_auc = roc_auc_score(image_labels, image_scores)
    except ValueError:
        image_auc = float('nan')

    results = {
        'pixel_f1_mean': float(np.mean(all_f1)),
        'pixel_f1_std': float(np.std(all_f1)),
        'pixel_iou_mean': float(np.mean(all_iou)),
        'pixel_iou_std': float(np.std(all_iou)),
        'precision_mean': float(np.mean(all_precision)),
        'recall_mean': float(np.mean(all_recall)),
        'tampered_f1_mean': float(np.mean(tampered_f1)) if tampered_f1 else 0.0,
        'tampered_f1_std': float(np.std(tampered_f1)) if tampered_f1 else 0.0,
        'tampered_iou_mean': float(np.mean(tampered_iou)) if tampered_iou else 0.0,
        'tampered_iou_std': float(np.std(tampered_iou)) if tampered_iou else 0.0,
        'image_accuracy': float(image_accuracy),
        'image_auc_roc': float(image_auc),
        'threshold_used': threshold,
        'num_test_images': len(all_f1),
        'num_tampered_images': len(tampered_f1),
        'forgery_breakdown': {},
        'size_stratification': {},
    }

    for ftype, scores in forgery_f1.items():
        if scores:
            results['forgery_breakdown'][ftype] = {
                'f1_mean': float(np.mean(scores)),
                'f1_std': float(np.std(scores)),
                'count': len(scores),
            }

    for bucket_name, bucket_info in size_buckets.items():
        scores = bucket_info['f1']
        if scores:
            results['size_stratification'][bucket_name] = {
                'f1_mean': float(np.mean(scores)),
                'f1_std': float(np.std(scores)),
                'count': len(scores),
            }

    return results


test_results = evaluate(model, test_loader, test_pairs, device, CONFIG, best_threshold)

# v8: Tampered-only metrics FIRST
print(f'\n{"=" * 60}')
print(f'TEST SET RESULTS (threshold={best_threshold:.4f})')
print(f'{"=" * 60}')

print(f'\nPRIMARY — Tampered-Only Localization ({test_results["num_tampered_images"]} images):')
print(f'  Pixel-F1:  {test_results["tampered_f1_mean"]:.4f} ± {test_results["tampered_f1_std"]:.4f}')
print(f'  Pixel-IoU: {test_results["tampered_iou_mean"]:.4f} ± {test_results["tampered_iou_std"]:.4f}')

print(f'\nBY FORGERY TYPE:')
for ftype, data in test_results['forgery_breakdown'].items():
    print(f'  {ftype} ({data["count"]} images): F1={data["f1_mean"]:.4f} ± {data["f1_std"]:.4f}')

print(f'\nBY MASK SIZE:')
for bucket_name, data in test_results['size_stratification'].items():
    print(f'  {bucket_name} ({data["count"]} images): F1={data["f1_mean"]:.4f} ± {data["f1_std"]:.4f}')

print(f'\nIMAGE-LEVEL DETECTION:')
print(f'  Accuracy: {test_results["image_accuracy"]:.4f}')
print(f'  AUC-ROC:  {test_results["image_auc_roc"]:.4f}')

print(f'\nSECONDARY — Mixed-Set ({test_results["num_test_images"]} images, includes authentic):')
print(f'  Pixel-F1:  {test_results["pixel_f1_mean"]:.4f} ± {test_results["pixel_f1_std"]:.4f}')
print(f'  Pixel-IoU: {test_results["pixel_iou_mean"]:.4f} ± {test_results["pixel_iou_std"]:.4f}')
print(f'  Precision: {test_results["precision_mean"]:.4f}')
print(f'  Recall:    {test_results["recall_mean"]:.4f}')

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


TEST SET RESULTS (threshold=0.7500)

PRIMARY — Tampered-Only Localization (769 images):
  Pixel-F1:  0.2949 ± 0.3450
  Pixel-IoU: 0.2321 ± 0.2956

BY FORGERY TYPE:
  splicing (274 images): F1=0.5758 ± 0.3278
  copy-move (495 images): F1=0.1394 ± 0.2400

BY MASK SIZE:
  tiny (<2%) (295 images): F1=0.1432 ± 0.2590
  small (2-5%) (180 images): F1=0.2429 ± 0.3139
  medium (5-15%) (152 images): F1=0.4057 ± 0.3463
  large (>15%) (142 images): F1=0.5573 ± 0.3446

IMAGE-LEVEL DETECTION:
  Accuracy: 0.7190
  AUC-ROC:  0.8170

SECONDARY — Mixed-Set (1893 images, includes authentic):
  Pixel-F1:  0.5181 ± 0.4621
  Pixel-IoU: 0.4926 ± 0.4615
  Precision: 0.5230
  Recall:    0.7579


In [26]:
if CONFIG['use_wandb']:
    wandb.summary.update({
        'best/val_f1': best_f1,
        'best/epoch': best_epoch + 1,
        'test/pixel_f1_tampered': test_results['tampered_f1_mean'],
        'test/pixel_f1_mixed': test_results['pixel_f1_mean'],
        'test/pixel_iou_mixed': test_results['pixel_iou_mean'],
        'test/image_accuracy': test_results['image_accuracy'],
        'test/image_auc_roc': test_results['image_auc_roc'],
        'test/threshold': best_threshold,
    })
    for ftype, data in test_results['forgery_breakdown'].items():
        wandb.summary[f'test/f1_{ftype}'] = data['f1_mean']
    print('Test results logged to W&B.')

Test results logged to W&B.


## 9. Visualization

1. **Training curves** — loss, F1, IoU, and LR over epochs
2. **F1 vs threshold** — expanded sweep visualization
3. **Prediction grid** — Original | GT Mask | Predicted Mask | Overlay

In [27]:
# ── Training Curves (v8: includes LR plot) ────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0, 0].plot(epochs_range, history['train_loss'], label='Train')
axes[0, 0].plot(epochs_range, history['val_loss'], label='Val')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, history['val_f1'], color='green')
axes[0, 1].axvline(x=best_epoch + 1, color='red', linestyle='--',
                    label=f'Best (epoch {best_epoch + 1})')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Pixel-F1')
axes[0, 1].set_title('Validation Pixel-F1')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, history['val_iou'], color='orange')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Pixel-IoU')
axes[1, 0].set_title('Validation Pixel-IoU')
axes[1, 0].grid(True, alpha=0.3)

# v8: LR schedule plot
axes[1, 1].plot(epochs_range, history['lr_encoder'], label='Encoder LR')
axes[1, 1].plot(epochs_range, history['lr_decoder'], label='Decoder LR')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].set_yscale('log')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

Training curves saved.


In [28]:
# ── F1 vs Threshold ──────────────────────────────────────────────────────────

thresh_vals = [r[0] for r in threshold_results]
f1_vals = [r[1] for r in threshold_results]
sorted_pairs_t = sorted(zip(thresh_vals, f1_vals))

plt.figure(figsize=(8, 5))
plt.plot([p[0] for p in sorted_pairs_t], [p[1] for p in sorted_pairs_t], 'b-', linewidth=2)
plt.axvline(x=best_threshold, color='red', linestyle='--',
            label=f'Best: {best_threshold:.3f} (F1={threshold_results[0][1]:.4f})')
plt.xlabel('Threshold'); plt.ylabel('Mean Pixel-F1')
plt.title('F1 vs. Threshold (Validation Set)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'f1_vs_threshold.png'), dpi=150, bbox_inches='tight')
plt.show()
print('F1 vs threshold plot saved.')

F1 vs threshold plot saved.


In [29]:
# ── Prediction Collection ─────────────────────────────────────────────────────

@torch.no_grad()
def collect_predictions(model, test_loader, test_pairs, device, config, threshold):
    """Collect all test predictions with metadata for visualization."""
    model.eval()
    predictions = []
    idx = 0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    for images, masks, labels in test_loader:
        images_dev = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            logits = model(images_dev)
        probs = torch.sigmoid(logits).cpu()

        for i in range(images.size(0)):
            pred_mask = (probs[i] > threshold).float()
            f1 = compute_pixel_f1(pred_mask, masks[i])
            img_np = images[i].permute(1, 2, 0).numpy() * std + mean
            img_np = np.clip(img_np, 0, 1)

            predictions.append({
                'image': img_np,
                'gt_mask': masks[i].squeeze().numpy(),
                'pred_mask': pred_mask.squeeze().numpy(),
                'prob_map': probs[i].squeeze().numpy(),
                'pixel_f1': f1,
                'label': labels[i].item(),
                'forgery_type': test_pairs[idx]['forgery_type'] if idx < len(test_pairs) else 'unknown',
                'gt_mask_area': masks[i].sum().item() / masks[i].numel(),
            })
            idx += 1

    return predictions


predictions = collect_predictions(model, test_loader, test_pairs, device, CONFIG, best_threshold)
print(f'Collected {len(predictions)} predictions.')

Collected 1893 predictions.


In [30]:
# ── Prediction Grid ───────────────────────────────────────────────────────────

tampered_preds = [p for p in predictions if p['label'] == 1.0]
authentic_preds = [p for p in predictions if p['label'] == 0.0]
tampered_sorted = sorted(tampered_preds, key=lambda p: p['pixel_f1'])

samples = []
if len(tampered_sorted) >= 2:
    samples.extend(tampered_sorted[-2:])  # Best
mid = len(tampered_sorted) // 2
if len(tampered_sorted) >= 4:
    samples.extend(tampered_sorted[mid - 1:mid + 1])  # Median
if len(tampered_sorted) >= 2:
    samples.extend(tampered_sorted[:2])  # Worst
if len(authentic_preds) >= 2:
    samples.extend(authentic_preds[:2])  # Authentic

n_rows = len(samples)
if n_rows == 0:
    print('No samples available for prediction grid.')
else:
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, 4 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row, sample in enumerate(samples):
        img, gt, pred = sample['image'], sample['gt_mask'], sample['pred_mask']
        overlay = img.copy()
        pred_bool = pred > 0
        if pred_bool.any():
            overlay[pred_bool] = overlay[pred_bool] * 0.6 + np.array([1, 0, 0]) * 0.4

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f'{sample["forgery_type"]} (F1={sample["pixel_f1"]:.3f})')
        axes[row, 0].axis('off')
        axes[row, 1].imshow(gt, cmap='gray', vmin=0, vmax=1)
        axes[row, 1].set_title('Ground Truth'); axes[row, 1].axis('off')
        axes[row, 2].imshow(pred, cmap='gray', vmin=0, vmax=1)
        axes[row, 2].set_title('Predicted Mask'); axes[row, 2].axis('off')
        axes[row, 3].imshow(overlay)
        axes[row, 3].set_title('Overlay'); axes[row, 3].axis('off')

    plt.suptitle('Prediction Grid: Best / Median / Worst / Authentic', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'prediction_grid.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Prediction grid saved.')

Prediction grid saved.


In [31]:
if CONFIG['use_wandb']:
    for plot_name in ['prediction_grid.png', 'training_curves.png', 'f1_vs_threshold.png']:
        plot_path = os.path.join(PLOTS_DIR, plot_name)
        if os.path.exists(plot_path):
            wandb.log({plot_name.replace('.png', ''): wandb.Image(plot_path)})
    print('Visualizations logged to W&B.')

Visualizations logged to W&B.


## 10. Explainable AI

- **Grad-CAM** — Spatial attention heatmaps from encoder layer4
- **Diagnostic overlays** — TP (green), FP (red), FN (blue) colour coding
- **Failure case analysis** — Worst-prediction breakdown with mask-size annotation

In [32]:
# ── Grad-CAM ──────────────────────────────────────────────────────────────────

class GradCAM:
    """Grad-CAM for segmentation encoder features."""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._handles = []
        self._handles.append(
            target_layer.register_forward_hook(self._save_activation)
        )
        self._handles.append(
            target_layer.register_full_backward_hook(self._save_gradient)
        )

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor):
        self.model.eval()
        self.gradients = None
        self.activations = None

        output = self.model(input_tensor)
        target = output.mean()
        self.model.zero_grad()
        target.backward()

        if self.gradients is None or self.activations is None:
            warnings.warn('Grad-CAM: hooks did not capture data.')
            return None

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        cam = F.interpolate(
            cam, size=input_tensor.shape[2:],
            mode='bilinear', align_corners=False
        )
        return cam.squeeze().cpu().numpy()

    def remove_hooks(self):
        for h in self._handles:
            h.remove()


def create_diagnostic_overlay(original, pred_mask, gt_mask):
    """Colour-coded overlay: Green=TP, Red=FP, Blue=FN."""
    overlay = original.copy().astype(np.float32)
    if overlay.max() > 1.0:
        overlay = overlay / 255.0

    tp_mask = (pred_mask > 0) & (gt_mask > 0)
    fp_mask = (pred_mask > 0) & (gt_mask == 0)
    fn_mask = (pred_mask == 0) & (gt_mask > 0)

    overlay[tp_mask] = overlay[tp_mask] * 0.5 + np.array([0, 1, 0]) * 0.5
    overlay[fp_mask] = overlay[fp_mask] * 0.5 + np.array([1, 0, 0]) * 0.5
    overlay[fn_mask] = overlay[fn_mask] * 0.5 + np.array([0, 0, 1]) * 0.5

    return np.clip(overlay, 0, 1)


print('GradCAM and diagnostic overlay defined.')

GradCAM and diagnostic overlay defined.


In [33]:
# ── Grad-CAM Visualization ────────────────────────────────────────────────────

_base = model.module if is_parallel else model
grad_cam = GradCAM(_base, _base.encoder.layer4)

cam_samples = [p for p in predictions if p['label'] == 1.0]
cam_samples = sorted(cam_samples, key=lambda p: p['pixel_f1'], reverse=True)[:4]

n_cam = len(cam_samples)
if n_cam == 0:
    print('No tampered samples for Grad-CAM visualization.')
else:
    fig, axes = plt.subplots(n_cam, 5, figsize=(25, 5 * n_cam))
    if n_cam == 1:
        axes = axes[np.newaxis, :]

    mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std_t = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    for row, sample in enumerate(cam_samples):
        img_tensor = torch.from_numpy(sample['image']).permute(2, 0, 1).float()
        img_tensor = (img_tensor - mean_t) / std_t
        img_tensor = img_tensor.unsqueeze(0).to(device)
        img_tensor.requires_grad_(True)

        try:
            cam = grad_cam.generate(img_tensor)
        except Exception as e:
            warnings.warn(f'Grad-CAM failed for row {row}: {e}')
            cam = None

        diagnostic = create_diagnostic_overlay(
            sample['image'], sample['pred_mask'], sample['gt_mask']
        )

        axes[row, 0].imshow(sample['image'])
        axes[row, 0].set_title(f'Original ({sample["forgery_type"]})')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(sample['image'])
        if cam is not None:
            axes[row, 1].imshow(cam, cmap='jet', alpha=0.5)
            axes[row, 1].set_title('Grad-CAM Heatmap')
        else:
            axes[row, 1].set_title('Grad-CAM (failed)')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(sample['gt_mask'], cmap='gray', vmin=0, vmax=1)
        axes[row, 2].set_title('Ground Truth'); axes[row, 2].axis('off')
        axes[row, 3].imshow(sample['pred_mask'], cmap='gray', vmin=0, vmax=1)
        axes[row, 3].set_title(f'Prediction (F1={sample["pixel_f1"]:.3f})')
        axes[row, 3].axis('off')
        axes[row, 4].imshow(diagnostic)
        axes[row, 4].set_title('Diagnostic (G=TP, R=FP, B=FN)')
        axes[row, 4].axis('off')

    plt.suptitle('Explainable AI: Grad-CAM + Diagnostic Overlays', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'gradcam_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Grad-CAM analysis saved.')

grad_cam.remove_hooks()

Grad-CAM analysis saved.


In [34]:
# ── Failure Case Analysis (v8: includes mask-size annotation) ─────────────────

def analyze_failure_cases(predictions, n_worst=10):
    """Analyze worst predictions to identify systematic error patterns."""
    tampered_preds = [p for p in predictions if p['label'] == 1.0]
    sorted_preds = sorted(tampered_preds, key=lambda p: p['pixel_f1'])
    worst = sorted_preds[:n_worst]

    if not worst:
        print('No tampered predictions to analyze.')
        return None

    analysis = {
        'forgery_types': [p['forgery_type'] for p in worst],
        'mean_mask_area': np.mean([p['gt_mask_area'] for p in worst]),
        'mean_f1': np.mean([p['pixel_f1'] for p in worst]),
        'common_patterns': [],
    }

    small_mask_count = sum(1 for p in worst if p['gt_mask_area'] < 0.02)
    if small_mask_count > n_worst // 2:
        analysis['common_patterns'].append(
            f'Fails on small tampered regions (<2% area): {small_mask_count}/{len(worst)}'
        )

    type_counts = Counter(analysis['forgery_types'])
    dominant_type = type_counts.most_common(1)[0]
    if dominant_type[1] > len(worst) * 0.7:
        analysis['common_patterns'].append(
            f'Disproportionately fails on {dominant_type[0]}: {dominant_type[1]}/{len(worst)}'
        )

    print(f'\nFailure Case Analysis (worst {len(worst)} predictions):')
    print(f'  Mean Pixel-F1:     {analysis["mean_f1"]:.4f}')
    print(f'  Mean GT mask area: {analysis["mean_mask_area"]:.4f}')
    print(f'  Forgery types:     {dict(type_counts)}')

    # v8: Per-case detail
    print(f'\n  Per-case breakdown:')
    for i, p in enumerate(worst):
        print(f'    #{i+1}: F1={p["pixel_f1"]:.3f}, type={p["forgery_type"]}, '
              f'mask_area={p["gt_mask_area"]:.4f} ({p["gt_mask_area"]*100:.1f}%)')

    if analysis['common_patterns']:
        print('\n  Patterns detected:')
        for pattern in analysis['common_patterns']:
            print(f'    - {pattern}')
    else:
        print('\n  No dominant failure patterns detected.')

    return analysis


failure_analysis = analyze_failure_cases(predictions)


Failure Case Analysis (worst 10 predictions):
  Mean Pixel-F1:     0.0000
  Mean GT mask area: 0.0214
  Forgery types:     {'copy-move': 9, 'splicing': 1}

  Per-case breakdown:
    #1: F1=0.000, type=copy-move, mask_area=0.0324 (3.2%)
    #2: F1=0.000, type=copy-move, mask_area=0.0116 (1.2%)
    #3: F1=0.000, type=splicing, mask_area=0.0140 (1.4%)
    #4: F1=0.000, type=copy-move, mask_area=0.0012 (0.1%)
    #5: F1=0.000, type=copy-move, mask_area=0.0216 (2.2%)
    #6: F1=0.000, type=copy-move, mask_area=0.0142 (1.4%)
    #7: F1=0.000, type=copy-move, mask_area=0.0025 (0.2%)
    #8: F1=0.000, type=copy-move, mask_area=0.0920 (9.2%)
    #9: F1=0.000, type=copy-move, mask_area=0.0124 (1.2%)
    #10: F1=0.000, type=copy-move, mask_area=0.0127 (1.3%)

  Patterns detected:
    - Fails on small tampered regions (<2% area): 7/10
    - Disproportionately fails on copy-move: 9/10


## 11. Robustness Testing

Evaluate under degradation conditions: JPEG compression, Gaussian noise, blur, resize.

**v8 expectation:** With JPEG/noise augmentation in training, the robustness gap should shrink from ~13% (Run01) to ~5%.

In [35]:
# ── Robustness Transforms ─────────────────────────────────────────────────────

NORMALIZE = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

robustness_transforms = {
    'clean': A.Compose([
        A.Resize(CONFIG['image_size'], CONFIG['image_size']),
        NORMALIZE, ToTensorV2(),
    ]),
    'jpeg_qf70': A.Compose([
        A.Resize(CONFIG['image_size'], CONFIG['image_size']),
        A.ImageCompression(quality_lower=70, quality_upper=70, p=1.0),
        NORMALIZE, ToTensorV2(),
    ]),
    'jpeg_qf50': A.Compose([
        A.Resize(CONFIG['image_size'], CONFIG['image_size']),
        A.ImageCompression(quality_lower=50, quality_upper=50, p=1.0),
        NORMALIZE, ToTensorV2(),
    ]),
    'gaussian_noise_light': A.Compose([
        A.Resize(CONFIG['image_size'], CONFIG['image_size']),
        A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
        NORMALIZE, ToTensorV2(),
    ]),
    'gaussian_noise_heavy': A.Compose([
        A.Resize(CONFIG['image_size'], CONFIG['image_size']),
        A.GaussNoise(var_limit=(100.0, 100.0), p=1.0),
        NORMALIZE, ToTensorV2(),
    ]),
    'gaussian_blur': A.Compose([
        A.Resize(CONFIG['image_size'], CONFIG['image_size']),
        A.GaussianBlur(blur_limit=(5, 5), p=1.0),
        NORMALIZE, ToTensorV2(),
    ]),
}


class ResizeDegradationDataset(Dataset):
    """Applies resize degradation. Masks remain clean."""

    def __init__(self, pairs, scale_factor, image_size=384):
        self.pairs = pairs
        self.scale_factor = scale_factor
        self.image_size = image_size
        self.normalize = A.Compose([
            A.Resize(image_size, image_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        entry = self.pairs[idx]
        image = cv2.imread(entry['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]
        small_h = max(1, int(h * self.scale_factor))
        small_w = max(1, int(w * self.scale_factor))
        degraded = cv2.resize(image, (small_w, small_h), interpolation=cv2.INTER_LINEAR)
        degraded = cv2.resize(degraded, (w, h), interpolation=cv2.INTER_LINEAR)

        if entry['mask_path'] is not None:
            mask = cv2.imread(entry['mask_path'], cv2.IMREAD_GRAYSCALE)
            mask = (mask > 0).astype(np.uint8)
        else:
            mask = np.zeros((h, w), dtype=np.uint8)

        augmented = self.normalize(image=degraded, mask=mask)
        image_t = augmented['image']
        mask_t = augmented['mask'].unsqueeze(0).float()
        label = torch.tensor(entry['label'], dtype=torch.float32)
        return image_t, mask_t, label


print(f'Defined {len(robustness_transforms)} degradation transforms + resize variants.')

Defined 6 degradation transforms + resize variants.


In [36]:
# ── Run Robustness Evaluation ─────────────────────────────────────────────────

@torch.no_grad()
def run_robustness_eval(model, loader, device, config, threshold):
    model.eval()
    f1_scores = []
    for images, masks, labels in loader:
        images = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            logits = model(images)
        probs = torch.sigmoid(logits).cpu()
        preds = (probs > threshold).float()
        for i in range(images.size(0)):
            f1_scores.append(compute_pixel_f1(preds[i], masks[i]))
    return f1_scores


def evaluate_robustness(model, test_pairs, device, config, threshold):
    results = {}

    for name, transform in tqdm(robustness_transforms.items(), desc='Robustness tests'):
        dataset = TamperingDataset(test_pairs, transform=transform)
        loader = DataLoader(
            dataset, batch_size=config['batch_size'],
            shuffle=False, num_workers=config['num_workers']
        )
        f1_scores = run_robustness_eval(model, loader, device, config, threshold)
        results[name] = {
            'f1_mean': float(np.mean(f1_scores)),
            'f1_std': float(np.std(f1_scores)),
        }

    for scale in [0.75, 0.5]:
        name = f'resize_{scale}x'
        dataset = ResizeDegradationDataset(test_pairs, scale_factor=scale)
        loader = DataLoader(
            dataset, batch_size=config['batch_size'],
            shuffle=False, num_workers=config['num_workers']
        )
        f1_scores = run_robustness_eval(model, loader, device, config, threshold)
        results[name] = {
            'f1_mean': float(np.mean(f1_scores)),
            'f1_std': float(np.std(f1_scores)),
        }

    return results


robustness_results = evaluate_robustness(model, test_pairs, device, CONFIG, best_threshold)

print(f'\nRobustness Results (threshold={best_threshold:.4f}):')
print(f'{"":<25} {"Pixel-F1 (mean ± std)":<25} {"Δ from clean":<15}')
print('-' * 65)
clean_f1 = robustness_results.get('clean', {}).get('f1_mean', 0)
for name, data in robustness_results.items():
    delta = data['f1_mean'] - clean_f1 if name != 'clean' else 0.0
    delta_str = f'{delta:+.4f}' if name != 'clean' else '---'
    print(f'{name:<25} {data["f1_mean"]:.4f} ± {data["f1_std"]:.4f}      {delta_str}')

# v8: Flag shortcut concern
jpeg_gap = abs(robustness_results.get('jpeg_qf50', {}).get('f1_mean', 0) - clean_f1)
if jpeg_gap > 0.10:
    print(f'\n⚠ JPEG robustness gap ({jpeg_gap:.3f}) exceeds 0.10 — shortcut learning risk persists.')
else:
    print(f'\n✓ JPEG robustness gap ({jpeg_gap:.3f}) is within acceptable range.')

Robustness tests:   0%|          | 0/6 [00:00<?, ?it/s]


Robustness Results (threshold=0.7500):
                          Pixel-F1 (mean ± std)     Δ from clean   
-----------------------------------------------------------------
clean                     0.5181 ± 0.4621      ---
jpeg_qf70                 0.5338 ± 0.4675      +0.0157
jpeg_qf50                 0.5092 ± 0.4698      -0.0090
gaussian_noise_light      0.3878 ± 0.4587      -0.1303
gaussian_noise_heavy      0.3894 ± 0.4604      -0.1287
gaussian_blur             0.4755 ± 0.4624      -0.0426
resize_0.75x              0.4650 ± 0.4592      -0.0531
resize_0.5x               0.4731 ± 0.4619      -0.0450

✓ JPEG robustness gap (0.009) is within acceptable range.


In [37]:
# ── Robustness Chart ──────────────────────────────────────────────────────────

names = list(robustness_results.keys())
means = [robustness_results[n]['f1_mean'] for n in names]
stds = [robustness_results[n]['f1_std'] for n in names]

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(names)), means, yerr=stds, capsize=4, color='steelblue', alpha=0.8)
if 'clean' in names:
    bars[names.index('clean')].set_color('green')

plt.xticks(range(len(names)), names, rotation=45, ha='right')
plt.ylabel('Pixel-F1')
plt.title('Robustness Testing: Pixel-F1 Under Degradation')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'robustness_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

if CONFIG['use_wandb']:
    wandb.log({'robustness_chart': wandb.Image(os.path.join(PLOTS_DIR, 'robustness_chart.png'))})
    for name, data in robustness_results.items():
        wandb.log({f'robustness/{name}_f1': data['f1_mean']})
    print('Robustness results logged to W&B.')

Robustness results logged to W&B.


## 12. Shortcut Learning Checks

Optional evaluation utilities to verify the model is not relying on dataset artifacts.

1. **Mask Randomization Test** — Feed random masks through evaluation; F1 should be ~0.0
2. **Boundary Sensitivity** — Measure erosion/dilation impact on F1

In [38]:
# ── Shortcut Learning Checks ──────────────────────────────────────────────────

@torch.no_grad()
def mask_randomization_test(model, test_loader, device, config, threshold, n_batches=10):
    """Feed predictions against random masks. F1 should be near 0.
    If F1 is high, the model may be predicting masks that correlate with
    dataset structure rather than actual tampering evidence.
    """
    model.eval()
    f1_scores = []
    batch_count = 0

    for images, masks, labels in test_loader:
        if batch_count >= n_batches:
            break
        images = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            logits = model(images)
        probs = torch.sigmoid(logits).cpu()
        preds = (probs > threshold).float()

        for i in range(images.size(0)):
            # Compare prediction against random binary mask
            random_mask = torch.randint(0, 2, masks[i].shape).float()
            f1 = compute_pixel_f1(preds[i], random_mask)
            f1_scores.append(f1)

        batch_count += 1

    mean_f1 = np.mean(f1_scores)
    print(f'Mask Randomization Test: F1 = {mean_f1:.4f} (expected: ~0.0-0.1)')
    if mean_f1 > 0.15:
        print('  ⚠ Unexpectedly high — model predictions may correlate with dataset structure.')
    else:
        print('  ✓ Predictions are not trivially correlated with random masks.')
    return mean_f1


def boundary_sensitivity_analysis(predictions, n_samples=50):
    """Measure how F1 changes with erosion/dilation of predictions.
    Stable F1 under small morphological changes suggests robust localization.
    """
    tampered = [p for p in predictions if p['label'] == 1.0 and p['pixel_f1'] > 0.1]
    tampered = tampered[:n_samples]

    if not tampered:
        print('No tampered predictions with F1 > 0.1 for boundary analysis.')
        return None

    kernel = np.ones((3, 3), np.uint8)
    results = {'original': [], 'eroded': [], 'dilated': []}

    for p in tampered:
        gt = p['gt_mask']
        pred = p['pred_mask']
        pred_u8 = pred.astype(np.uint8)

        results['original'].append(compute_pixel_f1(
            torch.from_numpy(pred).unsqueeze(0),
            torch.from_numpy(gt).unsqueeze(0)
        ))

        eroded = cv2.erode(pred_u8, kernel, iterations=2).astype(np.float32)
        results['eroded'].append(compute_pixel_f1(
            torch.from_numpy(eroded).unsqueeze(0),
            torch.from_numpy(gt).unsqueeze(0)
        ))

        dilated = cv2.dilate(pred_u8, kernel, iterations=2).astype(np.float32)
        results['dilated'].append(compute_pixel_f1(
            torch.from_numpy(dilated).unsqueeze(0),
            torch.from_numpy(gt).unsqueeze(0)
        ))

    print(f'\nBoundary Sensitivity Analysis ({len(tampered)} samples):')
    for variant, scores in results.items():
        print(f'  {variant:>10}: F1 = {np.mean(scores):.4f} ± {np.std(scores):.4f}')

    delta_erode = np.mean(results['original']) - np.mean(results['eroded'])
    delta_dilate = np.mean(results['dilated']) - np.mean(results['original'])
    print(f'  Erosion Δ: {delta_erode:+.4f}  |  Dilation Δ: {delta_dilate:+.4f}')

    return results


_ = mask_randomization_test(model, test_loader, device, CONFIG, best_threshold)
_ = boundary_sensitivity_analysis(predictions)

Mask Randomization Test: F1 = 0.0772 (expected: ~0.0-0.1)
  ✓ Predictions are not trivially correlated with random masks.

Boundary Sensitivity Analysis (50 samples):
    original: F1 = 0.5818 ± 0.2594
      eroded: F1 = 0.5857 ± 0.2679
     dilated: F1 = 0.5649 ± 0.2503
  Erosion Δ: -0.0040  |  Dilation Δ: -0.0169


## 13. Experiment Tracking

W&B tracking is integrated throughout, controlled by `CONFIG['use_wandb']`:

| Section | W&B Action |
|---|---|
| 2 (Setup) | Init with full CONFIG |
| 7 (Training) | Per-epoch: loss, F1, IoU, LR, grad_norm |
| 8 (Evaluation) | Test results summary, forgery breakdown |
| 9 (Visualization) | Plot images |
| 11 (Robustness) | Per-degradation F1 |
| 14 (Artifacts) | Model artifact upload, `wandb.finish()` |

## 14. Save Artifacts

Save all results, config, and upload model artifact to W&B if enabled.

In [39]:
# ── Save Results Summary ──────────────────────────────────────────────────────

results_summary = {
    'version': 'v8',
    'config': CONFIG,
    'seed': SEED,
    'best_epoch': best_epoch + 1,
    'best_val_f1': float(best_f1),
    'threshold': float(best_threshold),
    'pos_weight': float(pos_weight.item()) if pos_weight is not None else None,
    'test_results': {
        k: v for k, v in test_results.items()
        if k not in ('forgery_breakdown', 'size_stratification')
    },
    'forgery_breakdown': test_results.get('forgery_breakdown', {}),
    'size_stratification': test_results.get('size_stratification', {}),
    'robustness_results': {
        name: {'f1_mean': data['f1_mean'], 'f1_std': data['f1_std']}
        for name, data in robustness_results.items()
    },
    'training_history': {
        'epochs': len(history['train_loss']),
        'final_train_loss': history['train_loss'][-1] if history['train_loss'] else None,
        'final_val_f1': history['val_f1'][-1] if history['val_f1'] else None,
        'final_lr_encoder': history['lr_encoder'][-1] if history['lr_encoder'] else None,
        'final_lr_decoder': history['lr_decoder'][-1] if history['lr_decoder'] else None,
    },
    'v8_improvements': {
        'pos_weight': CONFIG['use_pos_weight'],
        'scheduler': CONFIG['scheduler'],
        'per_sample_dice': CONFIG['dice_per_sample'],
        'aug_color_jitter': CONFIG['aug_color_jitter'],
        'aug_compression': CONFIG['aug_compression'],
        'aug_gauss_noise': CONFIG['aug_gauss_noise'],
        'aug_gauss_blur': CONFIG['aug_gauss_blur'],
    },
}

results_path = os.path.join(RESULTS_DIR, 'results_summary.json')
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)

print(f'Results summary saved to: {results_path}')

Results summary saved to: /kaggle/working/results/results_summary.json


In [40]:
# ── W&B Artifact Upload ───────────────────────────────────────────────────────

if CONFIG['use_wandb']:
    artifact = wandb.Artifact('best-model-v8', type='model')
    best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
    if os.path.exists(best_model_path):
        artifact.add_file(best_model_path)
        wandb.log_artifact(artifact)
        print('Model artifact uploaded to W&B.')
    wandb.finish()
    print('W&B run finished.')

wandb: uploading artifact best-model-v8; updating run metadata


Model artifact uploaded to W&B.


wandb: uploading artifact best-model-v8; uploading config.yaml
wandb: uploading artifact best-model-v8
wandb: uploading summary, console lines 283-283
wandb: 
wandb: Run history:
wandb:                              epoch ▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
wandb:                robustness/clean_f1 ▁
wandb:        robustness/gaussian_blur_f1 ▁
wandb: robustness/gaussian_noise_heavy_f1 ▁
wandb: robustness/gaussian_noise_light_f1 ▁
wandb:            robustness/jpeg_qf50_f1 ▁
wandb:            robustness/jpeg_qf70_f1 ▁
wandb:          robustness/resize_0.5x_f1 ▁
wandb:         robustness/resize_0.75x_f1 ▁
wandb:                    train/grad_norm ▁▂▂▃▄▅▅▆▅▅▆▆▅▆▆▆▇██▆▇▇▆▇▇▇█
wandb:                                 +6 ...
wandb: 
wandb: Run summary:
wandb:                         best/epoch 17
wandb:                        best/val_f1 0.35847
wandb:                              epoch 27
wandb:                robustness/clean_f1 0.51811
wandb:        robustness/gaussian_blur_f1 0.4755
wandb: robustnes

W&B run finished.


In [41]:
# ── Artifact Inventory ────────────────────────────────────────────────────────

print()
print('=' * 60)
print('NOTEBOOK v8 COMPLETE — ARTIFACT INVENTORY')
print('=' * 60)

expected_artifacts = {
    CHECKPOINT_DIR: ['best_model.pt', 'last_checkpoint.pt'],
    RESULTS_DIR: ['split_manifest.json', 'results_summary.json'],
    PLOTS_DIR: ['training_curves.png', 'f1_vs_threshold.png', 'prediction_grid.png',
                'gradcam_analysis.png', 'robustness_chart.png'],
}

all_ok = True
for directory, files in expected_artifacts.items():
    print(f'\n{directory}/')
    for f_name in files:
        fpath = os.path.join(directory, f_name)
        status = 'OK' if os.path.exists(fpath) else 'MISSING'
        if status == 'MISSING':
            all_ok = False
        print(f'  [{status}] {f_name}')

print('\n' + '=' * 60)
if all_ok:
    print('All artifacts saved successfully.')
else:
    print('WARNING: Some artifacts are missing.')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Multi-GPU used:   {is_parallel}')
print(f'AMP used:         {CONFIG["use_amp"]}')
print(f'pos_weight:       {pos_weight.item() if pos_weight is not None else "disabled"}')
print(f'Scheduler:        {CONFIG["scheduler"]}')
print('=' * 60)


NOTEBOOK v8 COMPLETE — ARTIFACT INVENTORY

/kaggle/working/checkpoints/
  [OK] best_model.pt
  [OK] last_checkpoint.pt

/kaggle/working/results/
  [OK] split_manifest.json
  [OK] results_summary.json

/kaggle/working/plots/
  [OK] training_curves.png
  [OK] f1_vs_threshold.png
  [OK] prediction_grid.png
  [OK] gradcam_analysis.png
  [OK] robustness_chart.png

All artifacts saved successfully.
Output directory: /kaggle/working
Multi-GPU used:   True
AMP used:         True
pos_weight:       30.011716842651367
Scheduler:        reduce_on_plateau
